In [2]:
from __future__ import division
# import Message as ms
import cv2
import time
import numpy as np
import math

cap = cv2.VideoCapture(1)

# 设置摄像头分辨率为（640，480）
# 如果感觉图像卡顿严重，可以降低为（320，240）
#cap.set(3, 480)
#cap.set(4, 320)


# 设置黄色的阙值
yl = np.array([18, 127, 107])       #（36.50.42）
yu = np.array([23, 224, 231])       #（46.88.19）
#设置黑色阈值
bl = np.array([0,0,0])       #（0.0.0）
bu = np.array([188,255,46])    #（188，255，46）
#time.sleep(1)
flag_P=False
flag_Land=False
sum=0

while 1:
    # ret为是否找到图像， frame是帧本身
    ret, frame = cap.read()
    #frame = cv2.imread("F://2.jpg")
    ret=frame.copy()
    frame = cv2.GaussianBlur(frame, (5, 5), 0)  # 高斯模糊
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)  # 转hsv
    mask = cv2.inRange(hsv, bl, bu)  # 生成掩膜,第一个参数是原图，第2、3个是下、上阈值。这个阈值之间的->1，其他->0

    # 形态学操作
    mask = cv2.erode(mask, None, iterations=2)#腐蚀操作，值越大腐蚀越严重，使图像中的高亮区逐渐减小
    mask = cv2.dilate(mask, None, iterations=2)#膨胀操作，使图像中的高亮区域逐渐增长
    mask = cv2.GaussianBlur(mask, (3, 3), 0)
    res = cv2.bitwise_and(frame, frame, mask=mask)  # 与运算？？
    cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL,
                            cv2.CHAIN_APPROX_SIMPLE)[-2]  # 检测颜色的轮廓
    
    if len(cnts) > 0:
        cnt = max(cnts, key=cv2.contourArea)#   ???
        flag_P=True
        if cv2.contourArea(cnt)>500:
            (x, y), radius = cv2.minEnclosingCircle(cnt)
            cv2.circle(frame, (int(x), int(y)), int(radius) ,
                   (255, 0, 255), 2) # 找到后在每个轮廓上画圆
            d=frame.shape
            #print(d[0]/2)
            cx=int(d[1]/2)
            cy=int(d[0]/2)
            dx=int(x)-cx
            dy=int(y)-cy #待调整正负
            dis=math.sqrt(dx*dx+dy*dy)
            if(dis<50):
                sum+=1
            else:
                sum=0
            if(sum>10):
                flag_Land=True
            else:
                flag_Land=False
            # 正确的写法
            #pts = np.array([[x,y],[cx,cy]],np.int32)
            #cv2.polylines(img, [pts], isClosed, color,thickness)
            cv2.line(frame,(cx,cy),(int(x),int(y)),(150,250),3)
            #cv2.line(frame,(0,0),(abs(dx),abs(dy)),(150,250),3)
            #print('dx:',dx,'dy:',dy)
            angle=0
            if dx==0 :
                if(dy>=0):
                    angle=0
                else:
                    angle=3.14159
            elif(dy==0):
                if(dx>=0):
                    angle=1.570795
                else:
                    angle=-1.570795
            else:
                if(dx>0 and dy>0):
                    angle=math.atan(dx/dy)# 目标中心点相对于y轴正方向的偏角
                elif(dx>0 and dy<0):
                    angle=1.570795+math.atan(dx/dy)# 目标中心点相对于y轴正方向的偏角
                elif(dx<0 and dy<0):
                    angle=-1.570795-math.atan(dx/dy)# 目标中心点相对于y轴正方向的偏角
                elif(dx<0 and dy>0):
                    angle=math.atan(dx/dy)# 目标中心点相对于y轴正方向的偏角

            angle=angle*57.32# 偏角转换成角度制
            angletxt='angle is    '+str(angle)
            dtxt='dx: '+str(dx)+'  '+'dy: '+str(dy)+'  dis:   '+str(dis)
            flag='P: '+str(flag_P)+'   Land: '+str(flag_Land)
            #cv2.putText（img(图片文件名称),text(string),org(文字位置),
                            #fontFace(字体)，fontScale(文字大小),color(文字颜色),(文字粗细)）
            cv2.putText(frame,angletxt,(0,30),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            cv2.putText(frame,dtxt,(0,65),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            cv2.putText(frame,flag,(0,90),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
        #print('x:', x, 'y:', y)
#         Message
#         Message.UartSendData(Message.DotDataPack(c.color,c.flag,int(c.angle),int(c.distance),Message.Ctr.T_ms))
    else:
        flag_P=0
        
        
        
    cv2.namedWindow("capture", 0);
    cv2.resizeWindow("capture", 640, 480);
    cv2.imshow('capture', frame)
    if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
        break
cap.release()
cv2.destroyAllWindows()


In [1]:
# !/usr/bin/env python
# -*-conding:utf-8 -8-

"""
# File         : find_line.py
# Time         : 2023/4/29 15:16
# Author       : Shan
# email        : 1690942695@qq.com
# Description:
"""

import numpy as np

import cv2
import math
from matplotlib import pyplot as plt

# import adjust

# 霍夫直线检测阈值初始化
# threshold_HL = adjust.h.threshold
# minLineLength = adjust.h.minLineLength
# maxLineGap = adjust.h.maxLineGap
threshold_HL = 17
minLineLength = 1200
maxLineGap = 5

# fourcc = cv2.VideoWriter_fourcc(*'MJPG')
# out = cv2.VideoWriter('find_L2.mp4', fourcc, 2.0, (640,  480))

# PID控制参数
kp = 0.5
ki = 0.0
kd = 0.1

# 初始化PID参数
integral = 0.0
prev_error = 0.0

# 相机参数
# camera_matrix = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]])
# dist_coeffs = np.array([0, 0, 0, 0])

# 设置摄像头
cap = cv2.VideoCapture(1)

# 设置窗口大小为（320，240）
# cap.set(3, 320)
# cap.set(4, 240)

while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    # 进行Canny边缘检测                                        ???????有待改进
    edges = cv2.Canny(blur, 100, 200)
   


    edges = cv2.dilate(edges, None, iterations=10)#膨胀操作，使图像中的高亮区域逐渐增长
    edges = cv2.erode(edges, None, iterations=7)
    
    
    
    cv2.imshow('edges', edges)
    if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
        break
    # 进行霍夫直线检测
    
    
    
    
    
    
#                                                                          minLineLength  maxLineGap 
    lines = cv2.HoughLinesP(edges, 1,      np.pi / 180,            10,             10,          10)
# \                              rho     和theta的精度。  被视为一条线的最低投票数

# 第四个参数是阈值，这意味着它应该。请记住，投票数取决于线上的点数。
# 所以它可能代表了应该检测的最小线长。
#     - 线的最小长度。比这短的线段被拒绝。
#    - 线段之间的最大允许间隙，将它们视为一条线。






#     print(lines)
#     cv2.imshow('fram2', )
#     if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
#         break
    # 进行霍夫圆检测
    circles = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=50, param2=30, minRadius=10, maxRadius=50)

    if lines is not None:
#         将直线画出来
        i=0
        max_lenth=0
        
        
        
#         for line in lines:
#             x1,y1,x2,y2 = line[0]
#             lenth2=(x1-x2)*(x1-x2)+(y1-y2)*(y1-y2)
#             if(max_lenth<lenth2):
#                 max_lenth=lenth2



        for line in lines:
            x1,y1,x2,y2 = line[0]
#             lenth2=(x1-x2)*(x1-x2)+(y1-y2)*(y1-y2)
#             if(max_lenth==lenth2):
            cv2.line(frame,(x1,y1),(x2,y2),(0,255,0),2)
            p1='('+str(x1)+','+str(y1)+')'
            p2='('+str(x2)+','+str(y2)+')'
            #     cv2.putText（img(图片文件名称),text(string),org(文字位置),
#                             fontFace(字体)，fontScale(文字大小),color(文字颜色),(文字粗细)）
            cv2.putText(frame,p1,(0,30+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            cv2.putText(frame,p2,(0,65+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            i+=60
            break
#             rho, theta = line[0]
#             a = np.cos(theta)
#             b = np.sin(theta)
#             x0 = a * rho
#             y0 = b * rho
#             x1 = int(x0 + 1000 * (-b))
#             y1 = int(y0 + 1000 * (a))
#             x2 = int(x0 - 1000 * (-b))
#             y2 = int(y0 - 1000 * (a))
#             cv2.line(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
#         for line in lines:
#             x1 = line[0][0]
#                     #print(type(line))    <class 'numpy.intc'>
#             y1 = line[0][1]
#             x2 = line[0][2]
#             y2 = line[0][3]
#             cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # 计算偏差值
#         error = lines[0][0][1] - np.pi / 2

        # 计算PID控制量
#         integral += error
#         derivative = error - prev_error
#         control = kp * error + ki * integral + kd * derivative
#         prev_error = error

        # 计算目标直线与屏幕中心的直线距离
#         center_x = frame.shape[1] / 2
#         distance = abs((lines[0][0][0] - center_x) / np.cos(lines[0][0][1]))

        # 计算目标直线与y轴正方向的偏角
#         angle = lines[0][0][1] - np.pi / 2

        # 输出调试信息
        #print("控制量：%f，距离：%f，偏角：%f" % (control, distance, angle))

        # 如果找到了圆
#这里插入圆的判断
        # 显示结果

    cv2.imshow('frame', frame)
#     out.write(frame)
    if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
         break
cap.release()
cv2.destroyAllWindows()


error: OpenCV(4.5.5) D:\Build\OpenCV\opencv-4.5.5\modules\imgproc\src\color.cpp:182: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [4]:
import numpy as np
import cv2
from matplotlib import pyplot as plt

# import adjust

# 霍夫直线检测阈值初始化
# threshold_HL = adjust.h.threshold
# minLineLength = adjust.h.minLineLength
# maxLineGap = adjust.h.maxLineGap
threshold_HL = 17
minLineLength = 300
maxLineGap = 5

# PID控制参数
kp = 0.5
ki = 0.0
kd = 0.1

# 初始化PID参数
integral = 0.0
prev_error = 0.0


# 相机参数
# camera_matrix = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]])
# dist_coeffs = np.array([0, 0, 0, 0])

# 设置摄像头
cap = cv2.VideoCapture(1)

# 设置窗口大小为（320，240）
# cap.set(3, 320)
# cap.set(4, 240)

# 循环处理每一帧图像
while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 100, 200)
    edges = cv2.dilate(edges, None, iterations=9)#膨胀操作，使图像中的高亮区域逐渐增长
    edges = cv2.erode(edges, None, iterations=2)#腐蚀操作，值越大腐蚀越严重，使图像中的高亮区逐渐减小
    cv2.imshow('edges', edges)
    if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
        break
    # 进行霍夫直线检测
#     https://blog.csdn.net/u010087338/article/details/121580647?spm=1001.2014.3001.5506
# 第二个和第三个参数分别是rho 和theta的精度。
# 第四个参数是阈值，这意味着它应该被视为一条线的最低投票数。请记住，投票数取决于线上的点数。
# 所以它可能代表了应该检测的最小线长。





    lines = cv2.HoughLines(edges, 8, np.pi /11.25, 2400)
    # 进行霍夫圆检测
    
    
    
    
    
    
    circles = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=50, param2=30, minRadius=10, maxRadius=50)

    if lines is not None:
        # 将直线画出来
        i=0
        j=0
        for line in lines:
            rho, theta = line[0]
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho
            x1 = int(x0 + 1000 * (-b))
            y1 = int(y0 + 1000 * (a))
            x2 = int(x0 - 1000 * (-b))
            y2 = int(y0 - 1000 * (a))
            cv2.line(frame, (x1, y1), (x2, y2), (0, 0+i, 250+j), 2)
            p1='('+str(x1)+','+str(y1)+')'
            p2='('+str(x2)+','+str(y2)+')'
            #     cv2.putText（img(图片文件名称),text(string),org(文字位置),
#                             fontFace(字体)，fontScale(文字大小),color(文字颜色),(文字粗细)）
            cv2.putText(frame,p1,(0,30+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            cv2.putText(frame,p2,(0,65+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            i+=60
            j-=20
        # for line in lines:
        #     x1 = line[0][0]
        #     y1 = line[0][1]
        #     x2 = line[0][2]
        #     y2 = line[0][3]
        #     cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # 计算偏差值
        error = lines[0][0][1] - np.pi / 2

        # 计算PID控制量
        integral += error
        derivative = error - prev_error
        control = kp * error + ki * integral + kd * derivative
        prev_error = error

        # 计算目标直线与屏幕中心的直线距离
        center_x = frame.shape[1] / 2
        distance = abs((lines[0][0][0] - center_x) / np.cos(lines[0][0][1]))

        # 计算目标直线与y轴正方向的偏角
        angle = lines[0][0][1] - np.pi / 2

        # 输出调试信息
        #print("控制量：%f，距离：%f，偏角：%f" % (control, distance, angle))

        # 如果找到了圆
        
        #这里输入找圆程序
        
        # 显示结果
        cv2.imshow('frame', frame)

#         # 等待退出
        if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
            break
cap.release()
cv2.destroyAllWindows()



In [32]:
import numpy as np

import cv2
import math
from matplotlib import pyplot as plt

# import adjust

# 霍夫直线检测阈值初始化
# threshold_HL = adjust.h.threshold
# minLineLength = adjust.h.minLineLength
# maxLineGap = adjust.h.maxLineGap
threshold_HL = 17
minLineLength = 1200
maxLineGap = 5

# fourcc = cv2.VideoWriter_fourcc(*'MJPG')
# out = cv2.VideoWriter('find_L2.mp4', fourcc, 2.0, (640,  480))

# PID控制参数
kp = 0.5
ki = 0.0
kd = 0.1

# 初始化PID参数
integral = 0.0
prev_error = 0.0

# 相机参数
# camera_matrix = np.array([[600, 0, 320], [0, 600, 240], [0, 0, 1]])
# dist_coeffs = np.array([0, 0, 0, 0])

# 设置摄像头
cap = cv2.VideoCapture(1)

# 设置窗口大小为（320，240）
# cap.set(3, 320)
# cap.set(4, 240)

while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    # 进行Canny边缘检测                                        ???????有待改进
    edges = cv2.Canny(blur, 100, 200)
   


    edges = cv2.dilate(edges, None, iterations=10)#膨胀操作，使图像中的高亮区域逐渐增长
    edges = cv2.erode(edges, None, iterations=7)
    
    
    
    cv2.imshow('edges', edges)
    if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
        break
    # 进行霍夫直线检测
    
    
    
    
    
    
#                                                                          minLineLength  maxLineGap 
    lines = cv2.HoughLinesP(edges, 20,      np.pi / 180,            100,             900,          200)
# \                              rho     和theta的精度。  被视为一条线的最低投票数

# 第四个参数是阈值，这意味着它应该。请记住，投票数取决于线上的点数。
# 所以它可能代表了应该检测的最小线长。
#     - 线的最小长度。比这短的线段被拒绝。
#    - 线段之间的最大允许间隙，将它们视为一条线。






#     print(lines)
#     cv2.imshow('fram2', )
#     if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
#         break
    # 进行霍夫圆检测
    circles = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=50, param2=30, minRadius=10, maxRadius=50)

    if lines is not None:
        
        i=0
        max_lenth=0
        minAngle=math.pi/2
        maxAngle=0
        x11,x21,x12,x22,y11,y21,y12,y22=0,0,0,0,0,0,0,0
        
        
        
        
        for line in lines:
            x1,y1,x2,y2 = line[0]
            lenth2=(x1-x2)*(x1-x2)+(y1-y2)*(y1-y2)
            deltaY=y1-y2
            radian=deltaY/lenth2
            absAngle=math.fabs(math.asin(radian))
            if(maxAngle<absAngle):
                maxAngle=absAngle
                x11,x12,y11,y12=x1,x2,y1,y2
            elif(minAngle>absAngle):
                minAngle=absAngle
                x21,x22,y21,y22=x1,x2,y1,y2

                
                

            #画线1+显示文字
        for line in lines:
            cv2.line(frame,(x11,y11),(x12,y12),(0,255,0),2)
            cv2.line(frame,(x21,y21),(x22,y22),(0,0,255),2)
            p1='1st line  ('+str(x11)+','+str(y11)+')'
            p2='('+str(x12)+','+str(y12)+')'
            p3='2ed line  ('+str(x21)+','+str(y21)+')'
            p4='('+str(x22)+','+str(y22)+')'
            if((maxAngle-minAngle)>0.5):
                str='要转的角度=   '+str((maxAngle-minAngle)*57.3)
                cv2.putText(frame,str,(0,30+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
#                
            #     cv2.putText（img(图片文件名称),text(string),org(文字位置),
#                             fontFace(字体)，fontScale(文字大小),color(文字颜色),(文字粗细)）
            cv2.putText(frame,p1,(0,65+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),1)
            cv2.putText(frame,p2,(400,65+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),1)
            cv2.putText(frame,p3,(0,100+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            cv2.putText(frame,p4,(400,100+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),1)
            i+=60
            
            
            
            
            #画线+显示文字
#             cv2.line(frame,(x1,y1),(x2,y2),(0,255,0),2)
#             p1='1st line  ('+str(x1)+','+str(y1)+')'
#             p2='('+str(x2)+','+str(y2)+')'
#             cv2.putText(frame,p1,(0,65+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),1)
#             cv2.putText(frame,p2,(400,65+i),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),1)
#             i+=60
            


    cv2.imshow('frame', frame)
#     out.write(frame)
    if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
         break
cap.release()
cv2.destroyAllWindows()


In [26]:
import math
angle=math.sqrt(2)/2
print(math.asin(angle))

0.7853981633974484


In [ ]:
--------测量HSV值程序-------孟祥喆-------4.25


from __future__ import division
# import Message as ms
import cv2
import time
import numpy as np
import math

cap = cv2.VideoCapture(0)

# 设置摄像头分辨率为（640，480）
# 如果感觉图像卡顿严重，可以降低为（320，240）
#cap.set(3, 480)
#cap.set(4, 320)


# 设置黄色的阙值
yl = np.array([18, 127, 107])       #（36.50.42）
yu = np.array([23, 224, 231])       #（46.88.19）
#设置黑色阈值
bl = np.array([0,0,0])       #（0.0.0）
bu = np.array([188,255,46])    #（188，255，46）
#time.sleep(1)
flag_P=False
flag_Land=False
sum=0

while 1:
    # ret为是否找到图像， frame是帧本身
    ret, frame = cap.read()
    #frame = cv2.imread("F://2.jpg")
    ret=frame.copy()
    frame = cv2.GaussianBlur(frame, (5, 5), 0)  # 高斯模糊
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)  # 转hsv
    
    
        
        
        
    cv2.namedWindow("capture", 0);
    cv2.resizeWindow("capture", 640, 480);
    cv2.imshow('capture', frame)
    if cv2.waitKey(1)& 0xff==ord('q'):  #esc退出
        break
cap.release()
cv2.destroyAllWindows()